# Assignment 2

 **This was supoosed to be 2 assignment but is a big assignment So go slow and learn what you are doing have fun**


# Part 1: Data Ingestion

Data Ingestion is the first step in a RAG pipeline. It involves collecting, reading, and loading raw data from various sources (such as PDFs, HTML, or databases) into the system. Here, we read all PDF files in a given directory and parse their content into structured documents containing page content and metadata.


Here we Doing only for pdf but in final project we will do for pdf,csv,xlsx,docx,txt.
if you want you can practise to extract data from one more file format i would love to see you do this.

In [1]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader



C:\Users\manth\AppData\Local\Temp\ipykernel_28616\1346314019.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [2]:

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    # TODO: Implement PDF ingestion using PyPDFLoader.
    # Load all pdf files recursively from pdf_directory (using Path(pdf_directory).glob('**/*.pdf')).
    # For each page document loaded, add metadata fields:
    # - 'source_file': the filename of the PDF file
    # - 'file_type': 'pdf'
    # Collect all loaded documents in a list and return them.
    # Hint: Use PyPDFLoader(str(pdf_file)).load()
    all_pdf_documents = []
    pdf_path = Path(pdf_directory)
    pdf_files = list(pdf_path.glob('**/*.pdf'))
    print(f"No of PDF Files :{len(pdf_files)} ")
    
    for pdf_file in pdf_files:
        print(f"Loading: {pdf_file.name}")
        try:

            loader = PyMuPDFLoader(str(pdf_file))
            pages = loader.load()
            for page in pages:
                page.metadata['source_file'] = pdf_file.name
                page.metadata['file_type'] = 'pdf'
                
            all_pdf_documents.extend(pages)
        except Exception as e:
            print(f"Error loading {pdf_file.name}: {e}")
            
    print(f"Total pages: {len(all_pdf_documents)} ")
    return all_pdf_documents

    


In [3]:
all_pdf_documents=process_all_pdfs("./pdf_directory")
first_three_pages = all_pdf_documents[:3]

for i, page in enumerate(first_three_pages):
    print(f"\n{'='*50}")
    print(f"📄 PAGE {i + 1} METADATA")
    print(f"{'='*50}")
    print(page.metadata)
    
    print(f"\n📝 PAGE {i + 1} TEXT PREVIEW (First 500 characters)")
    print("-" * 50)
    print(page.page_content[:500] + "...\n")

No of PDF Files :1 
Loading: NIPS-2017-attention-is-all-you-need-Paper.pdf
Total pages: 11 

📄 PAGE 1 METADATA
{'producer': 'PyPDF2', 'creator': '', 'creationdate': '', 'source': 'pdf_directory\\NIPS-2017-attention-is-all-you-need-Paper.pdf', 'file_path': 'pdf_directory\\NIPS-2017-attention-is-all-you-need-Paper.pdf', 'total_pages': 11, 'format': 'PDF 1.3', 'title': 'Attention is All you Need', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'keywords': '', 'moddate': '2018-02-12T21:22:10-08:00', 'trapped': '', 'modDate': "D:20180212212210-08'00'", 'creationDate': '', 'page': 0, 'source_file': 'NIPS-2017-attention-is-all-you-need-Paper.pdf', 'file_type': 'pdf'}

📝 PAGE 1 TEXT PREVIEW (First 500 characters)
--------------------------------------------------
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam

# Part 2: Chunking

Chunking is the process of breaking down large documents into smaller, cohesive segments (chunks). Since Large Language Models (LLMs) have a limited context window (input size limit) and retrieve information more accurately from smaller context blocks, we must split large documents. In this assignment, you need to use **RecursiveCharacterTextSplitter** to split loaded documents into smaller paragraphs with proper overlap to maintain context between boundary lines.


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [5]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    # TODO: Implement text splitting using RecursiveCharacterTextSplitter.
    # You need to use RecursiveCharacterTextSplitter with:
    # - chunk_size=chunk_size of your choice 
    # - chunk_overlap=chunk_overlap (recommended 20% of chunk_size)
    # - length_function=len
    # - separators=['\n\n', '\n', ' ', '']
    # Print how many documents were split into how many chunks. (Tip: do not use too big files it will it time)
    # Return the split documents list.
    if not documents:
        print("No documents provided to split.")
        return []
        
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=['\n\n', '\n', ' ', '']
    )
    
    split_docs = splitter.split_documents(documents)
    print(f"Length of documents {len(documents)} ")
    print(f"No of chunks {len(split_docs)}")
    return split_docs
    


In [6]:
chunks=split_documents(all_pdf_documents)
chunks

Length of documents 11 
No of chunks 43


[Document(metadata={'producer': 'PyPDF2', 'creator': '', 'creationdate': '', 'source': 'pdf_directory\\NIPS-2017-attention-is-all-you-need-Paper.pdf', 'file_path': 'pdf_directory\\NIPS-2017-attention-is-all-you-need-Paper.pdf', 'total_pages': 11, 'format': 'PDF 1.3', 'title': 'Attention is All you Need', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'keywords': '', 'moddate': '2018-02-12T21:22:10-08:00', 'trapped': '', 'modDate': "D:20180212212210-08'00'", 'creationDate': '', 'page': 0, 'source_file': 'NIPS-2017-attention-is-all-you-need-Paper.pdf', 'file_type': 'pdf'}, page_content='Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Researc

# Part 3: Embedding

Embedding is the process of converting text blocks into numerical vector representations. These vectors capture the semantic meaning of the text. Sentences that are semantically similar will be closer to each other in the vector space. We use pre-trained sentence transformer models (like 'all-MiniLM-L6-v2') to convert text chunks and queries into embeddings.

---



from sentence_transformers import SentenceTransformer

Imports the embedding model class.

This library:

downloads pretrained transformer models
converts text → embeddings

Built on top of:

HuggingFace transformers
PyTorch

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid #to get each chunk a unique id after embedding
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        # TODO: Load the SentenceTransformer model using self.model_name.
        # Print a message stating the model name is being loaded.
        # Set self.model to the loaded SentenceTransformer.
        # Print a message indicating successful loading and print the embedding dimension.
        print(f"Loading SentenceTransformer model: '{self.model_name}'...")
        self.model = SentenceTransformer(self.model_name)
        test_vector = self.model.encode(["test"])
        dimension = test_vector.shape[1]
        print(f"Model successfully loaded. Embedding Dimension: {dimension}")
        

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        # TODO: Generate embeddings for the given list of texts using the model.
        # Make sure to handle cases where self.model is None.
        # Use self.model.encode(texts, show_progress_bar=True).
        # Return the resulting numpy array of embeddings.
        if self.model is None:
            raise ValueError("Model is not initialized. Cannot generate embeddings.")
            
        if not texts:
            return np.empty((0, 384)) 
        return self.model.encode(texts, show_progress_bar=True)
        
        



# Part 4: Vector DB

Vector Database (Vector DB) is a specialized database designed to store and query high-dimensional vector embeddings efficiently. It allows us to persist our document chunks along with their computed embeddings and perform fast search operations. Here, we use ChromaDB, which runs locally and stores documents persistently in a directory.

In [9]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        # TODO: Create the persist directory if it doesn't exist.
        # Initialize chromadb.PersistentClient with path=self.persist_directory.
        # Get or create a collection using self.client.get_or_create_collection
        # with self.collection_name and description metadata.
        os.makedirs(self.persist_directory, exist_ok=True)
        self.client = chromadb.PersistentClient(path=self.persist_directory)
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "Vector store for PDF document RAG assignment"}
        )
        print(f"ChromaDB collection '{self.collection_name}' initialized at {self.persist_directory}")

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        # TODO: Add documents to ChromaDB collection.
        # 1. Check if the number of documents matches the number of embeddings.
        # 2. For each document, generate a unique ID using uuid.uuid4().hex[:8]
        # 3. Construct metadata dict including original document metadata, doc_index, and content_length.
        # 4. Extract document content string.
        # 5. Convert embedding to a list using .tolist().
        # 6. Add ids, embeddings, metadatas, and documents to self.collection.
        if len(documents) != len(embeddings):
            raise ValueError(f"Mismatch: Got {len(documents)} documents but {len(embeddings)} embeddings.")
            
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = embeddings.tolist()
        
        for idx, doc in enumerate(documents):
            # 2. Generate unique hex identifier
            chunk_id = uuid.uuid4().hex[:8]
            ids.append(chunk_id)
            
            # 3. Construct dictionary parsing structural metadata
            meta = dict(doc.metadata) if doc.metadata else {}
            meta['doc_index'] = idx
            meta['content_length'] = len(doc.page_content)
            metadatas.append(meta)
            
            # 4. Extract content string
            documents_text.append(doc.page_content)
            
        # 6. Push vector records natively into collection
        if ids:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully injected {len(ids)} document chunks into Vector DB.")


## convert text to embeddings


In [10]:
### lets see text of chumks
texts=[doc.page_content for doc in chunks]

texts

['Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer,\nbased solely on attention mechanisms, dispensing with recurrence and convolutions\nentirely. Experiments on two machine translation tasks show these models to\nbe superior in quality while being more parallelizable and requiring signiﬁcantl

In [11]:
## Generate the Embeddings
embedding_manager = EmbeddingManager()
vectorstore = VectorStore()
embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Loading SentenceTransformer model: 'all-MiniLM-L6-v2'...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model successfully loaded. Embedding Dimension: 384
ChromaDB collection 'pdf_documents' initialized at ../data/vector_store


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Successfully injected 43 document chunks into Vector DB.


# Part 5: Query Retrieval

Query Retrieval starts with the user entering a natural language query. We must convert this query into the same embedding space using our embedding manager. This encoded query is then sent to the vector database to retrieve raw results.

---

# Part 6: Similarity Search

Similarity Search is the mathematical calculation (such as Cosine Similarity) used by the vector store to compare the query embedding with document embeddings. It ranks and returns the top_k most similar documents. We can filter results by a minimum similarity score (score_threshold) to keep only relevant context.


In [12]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        # TODO: Retrieve relevant documents from the ChromaDB collection.
        # 1. Generate query embedding using self.embedding_manager.generate_embeddings.
        # 2. Query the vector store collection with the query embedding list, asking for top_k results.
        # 3. For each result, convert distance to similarity score (similarity_score = 1 - distance).
        # 4. Filter results that have similarity_score >= score_threshold.
        # 5. Return a list of dicts with keys: 'id', 'content', 'metadata', 'similarity_score', 'distance', 'rank'.
        
        query_vector = self.embedding_manager.generate_embeddings([query])[0].tolist()
        results = self.vector_store.collection.query(
            query_embeddings=[query_vector],
            n_results=top_k
        )
        retrieved_items = []
        if not results or not results['ids'] or len(results['ids'][0]) == 0:
            return []
        ids = results['ids'][0]
        distances = results['distances'][0] if results['distances'] else [0.0] * len(ids)
        documents = results['documents'][0]
        metadatas = results['metadatas'][0]
        
        for rank, (c_id, dist, content, meta) in enumerate(zip(ids, distances, documents, metadatas), start=1):
            similarity_score = 1.0 - dist if dist <= 1.0 else 1.0 / (1.0 + dist)
            if similarity_score >= score_threshold:
                retrieved_items.append({
                    'id': c_id,
                    'content': content,
                    'metadata': meta,
                    'similarity_score': similarity_score,
                    'distance': dist,
                    'rank': rank
                })
                
        return retrieved_items


In [13]:
rag_retriever = RAGRetriever(vector_store=vectorstore, embedding_manager=embedding_manager)

In [14]:
rag_retriever.retrieve("Why does the Transformer model use positional encodings? ")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[{'id': 'abd6efdc',
  'content': 'Figure 1: The Transformer - model architecture.\nwise fully connected feed-forward network. We employ a residual connection [10] around each of\nthe two sub-layers, followed by layer normalization [1]. That is, the output of each sub-layer is\nLayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer\nitself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding\nlayers, produce outputs of dimension dmodel = 512.\nDecoder:\nThe decoder is also composed of a stack of N = 6 identical layers. In addition to the two\nsub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head\nattention over the output of the encoder stack. Similar to the encoder, we employ residual connections\naround each of the sub-layers, followed by layer normalization. We also modify the self-attention\nsub-layer in the decoder stack to prevent positions from attending to

# Part 7: Prompting and Calling LLM

Prompting and Calling LLM is the synthesis step of RAG. We take the retrieved contexts, format them into a structured prompt alongside the user's original query, and pass them to the Large Language Model (LLM) to generate a grounded, factually accurate response.


In [15]:
def rag_simple(query,retriever,llm,top_k=3):
    # TODO: Implement a simple RAG pipeline using following steps which join above functions.
    # 1. Use the retriever to fetch top_k documents for the query.
    # 2. Join the content of the retrieved documents to form the context.
    # 3. Format a prompt instructing the LLM to use the context to answer the question.
    # 4. Call the LLM with the formatted prompt and return the string content of the response.
    retrieved_docs = retriever.retrieve(query, top_k=top_k)
    context = "\n\n---\n\n".join([doc['content'] for doc in retrieved_docs])
    prompt = f"""You are an intelligent assistant. Use the following pieces of context to accurately answer the question at the end. 
If you do not know the answer based on the provided context, state clearly that the information is missing. Do not make up facts.

Context:
{context}

Question: {query}
Answer:"""

    if hasattr(llm, 'invoke'):
        response = llm.invoke(prompt)
        return response.content if hasattr(response, 'content') else str(response)
    else:
        return llm(prompt)
    


In [16]:
import os
from langchain_groq import ChatGroq
os.environ["GROQ_API_KEY"] = "Enter your Groq api key"
#Cannot enter my own api key becuase then git wont allow me to push to github
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.2,
    max_tokens=None,
    timeout=None,
    max_retries=2
)

print("Groq LLM wrapper initialized successfully!")

Groq LLM wrapper initialized successfully!


In [17]:
answer=rag_simple("what is attention?",rag_retriever,llm)
print(answer)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}

# Part 8: Advanced RAG (Optional)

Advanced RAG includes sophisticated pipeline elements such as streaming responses, citation tracking, interaction history (conversational memory), response summarization, and score-based filtering to make the application robust and production-ready.


In [75]:
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # TODO: Implement the AdvancedRAGPipeline query logic:
        # 1. Retrieve documents using self.retriever.
        # 2. Parse sources/metadata and construct the context.
        # 3. If stream=True, simulate streaming by printing the prompt in chunks.
        # 4. Generate the answer using self.llm.
        # 5. Construct citations list and append it to the answer.
        # 6. If summarize=True, call the LLM to get a 2-sentence summary of the answer.
        # 7. Append the question, answer, sources, and summary to self.history.
        # 8. Return dictionary containing question, answer (with citations), sources, summary, and history.
        docs = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        context = "\n\n".join([d['content'] for d in docs])
        source_files = list(set([d['metadata'].get('source_file', 'Unknown File') for d in docs]))
        
        prompt = f"Context:\n{context}\n\nQuestion: {question}\n\nDetailed Answer:"
        if stream:
            print(">>> DEBUG [Streaming Mock Prompt Construct]...")
            for chunk in [prompt[i:i+40] for i in range(0, len(prompt), 40)]:
                print(chunk, end="", flush=True)
            print("\n>>> End Stream Debug\n")
        if hasattr(self.llm, 'invoke'):
            res = self.llm.invoke(prompt)
            answer = res.content if hasattr(res, 'content') else str(res)
        else:
            answer = self.llm(prompt)
        citations = [f"[{idx+1}] {d['metadata'].get('source_file', 'Unknown')} (Page {d['metadata'].get('page', 'N/A')})" for idx, d in enumerate(docs)]
        if citations:
            answer += "\n\nReferences:\n" + "\n".join(citations)
        summary_text = "No summary requested."
        if summarize and docs:
            summary_prompt = f"Summarize this response into exactly two concise sentences:\n\n{answer}"
            if hasattr(self.llm, 'invoke'):
                s_res = self.llm.invoke(summary_prompt)
                summary_text = s_res.content if hasattr(s_res, 'content') else str(s_res)
            else:
                summary_text = self.llm(summary_prompt)
        record = {
            'question': question,
            'answer': answer,
            'sources': source_files,
            'summary': summary_text
        }
        self.history.append(record)
        record['history'] = self.history
        
        return record


In [82]:
advancedRAGPipeline = AdvancedRAGPipeline(rag_retriever,llm)
result=advancedRAGPipeline.query("what is the title of the document")
print(result['answer'])


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

The title of the document is not explicitly stated in the provided text. However, based on the content, it appears to be related to the Transformer model, a type of neural network architecture. The text mentions "the Transformer" and describes its characteristics, such as relying entirely on self-attention to compute representations of its input and output. 

A possible title for the document could be "Attention Is All You Need" or "The Transformer Model", as these titles are commonly associated with the introduction of the Transformer architecture in the field of natural language processing. However, without more context or information, it is impossible to determine the exact title of the document.

References:
[1] NIPS-2017-attention-is-all-you-need-Paper.pdf (Page 10)
[2] NIPS-2017-attention-is-all-you-need-Paper.pdf (Page 10)
[3] NIPS-2017-attention-is-all-you-need-Paper.pdf (Page 9)
[4] NIPS-2017-attention-is-all-you-need-Paper.pdf (Page 9)
[5] NIPS-2017-attention-is-all-you-need-